# Fase 1 · Datos y preprocesamiento (CIFAR-10)

**Objetivo de este notebook**
1. Cargar el dataset CIFAR-10.
2. Entender su estructura (tamaños, clases, distribución).
3. Visualizar imágenes con sus etiquetas.
4. Dividir los datos en **train / validation / test**.
5. Normalizar los píxeles.
6. Preparar el **data augmentation** y un pipeline eficiente con `tf.data`.

> 💡 **En Colab:** ve a *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)* antes de empezar.

## 1. Importar librerías y fijar la semilla

Fijar una **semilla** (seed) hace que los resultados aleatorios (la división de datos, la inicialización de pesos...) sean **reproducibles**: si lo ejecutas dos veces, obtienes lo mismo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)  # fija la semilla de Python, NumPy y TensorFlow

print("Versión de TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

## 2. Cargar CIFAR-10

Keras trae CIFAR-10 incorporado: lo descarga (unos 170 MB) la primera vez.

- `x_` son las **imágenes** (los datos de entrada).
- `y_` son las **etiquetas** (un número del 0 al 9 que indica la clase).

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Las etiquetas vienen con forma (N, 1); las aplanamos a (N,) para que sea más cómodo
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("Train completo:", x_train_full.shape, y_train_full.shape)
print("Test:          ", x_test.shape, y_test.shape)
print("Tipo de dato:", x_train_full.dtype, "| Rango de píxeles:", x_train_full.min(), "-", x_train_full.max())

**¿Cómo se lee `(50000, 32, 32, 3)`?**
- `50000` → número de imágenes
- `32, 32` → alto y ancho en píxeles
- `3` → canales de color (rojo, verde, azul = RGB)

Cada píxel es un número entero entre **0 y 255**.

## 3. Distribución de clases

Comprobamos si el dataset está **balanceado** (si todas las clases tienen el mismo número de imágenes). Si no lo estuviera, la accuracy podría engañarnos.

In [ ]:
clases, conteos = np.unique(y_train_full, return_counts=True)

plt.figure(figsize=(10, 4))
plt.bar([CLASS_NAMES[c] for c in clases], conteos, color="steelblue")
plt.title("Número de imágenes por clase (train)")
plt.ylabel("Imágenes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

for c, n in zip(clases, conteos):
    print(f"{CLASS_NAMES[c]:>11}: {n}")

✅ Las 10 clases tienen exactamente 5.000 imágenes: el dataset está **perfectamente balanceado**, así que la accuracy es una métrica fiable.

## 4. Visualizar imágenes con sus etiquetas

In [ ]:
def mostrar_imagenes(imagenes, etiquetas, n_filas=4, n_cols=8, titulo=None):
    """Muestra una cuadrícula de imágenes con el nombre de su clase encima.

    Args:
        imagenes: array de imágenes con forma (N, alto, ancho, 3).
        etiquetas: array de enteros (0-9) con la clase de cada imagen.
        n_filas, n_cols: tamaño de la cuadrícula.
        titulo: título general opcional.
    """
    plt.figure(figsize=(n_cols * 1.6, n_filas * 1.8))
    for i in range(n_filas * n_cols):
        plt.subplot(n_filas, n_cols, i + 1)
        plt.imshow(imagenes[i])
        plt.title(CLASS_NAMES[etiquetas[i]], fontsize=9)
        plt.axis("off")
    if titulo:
        plt.suptitle(titulo, fontsize=14)
    plt.tight_layout()
    plt.show()

# 32 imágenes elegidas al azar
idx = np.random.choice(len(x_train_full), 32, replace=False)
mostrar_imagenes(x_train_full[idx], y_train_full[idx], titulo="Ejemplos aleatorios de CIFAR-10")

Ahora un ejemplo de **cada clase** para hacernos una idea de lo que tiene que aprender el modelo:

In [ ]:
plt.figure(figsize=(15, 8))
for clase in range(10):
    idx_clase = np.where(y_train_full == clase)[0][:6]   # las 6 primeras de esa clase
    for j, i in enumerate(idx_clase):
        plt.subplot(6, 10, j * 10 + clase + 1)
        plt.imshow(x_train_full[i])
        plt.axis("off")
        if j == 0:
            plt.title(CLASS_NAMES[clase], fontsize=10)
plt.suptitle("6 ejemplos por clase", fontsize=14)
plt.tight_layout()
plt.show()

🔎 **Observación:** las imágenes son muy pequeñas (32x32), y algunas clases se parecen mucho entre sí (**cat** frente a **dog**, **automobile** frente a **truck**). Es de esperar que el modelo confunda esas parejas: lo comprobaremos con la matriz de confusión en la fase 4.

## 5. Dividir en train / validation / test

| Conjunto | Para qué sirve | Tamaño |
|---|---|---|
| **Train** | El modelo aprende con estos datos | 45.000 |
| **Validation** | Vigilar el entrenamiento (early stopping, elegir hiperparámetros) | 5.000 |
| **Test** | Evaluación **final**: solo se usa al final | 10.000 |

⚠️ **Error típico:** usar el conjunto de test para tomar decisiones durante el entrenamiento. Eso «filtra» información y da un resultado demasiado optimista.

Con `stratify` conseguimos que cada conjunto mantenga la misma proporción de clases.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=5000,
    stratify=y_train_full,
    random_state=SEED,
)

print("Train:     ", x_train.shape)
print("Validation:", x_val.shape)
print("Test:      ", x_test.shape)
print("Imágenes por clase en validation:", np.bincount(y_val))

## 6. Normalización

Las redes neuronales aprenden mejor con valores pequeños. Pasamos los píxeles de **[0, 255]** a **[0, 1]** dividiendo entre 255.

> 📝 Hacemos la normalización **dentro del modelo** (con una capa `Rescaling`). Así, en el despliegue (fase 6) le podremos pasar la imagen tal cual y el modelo se encargará de normalizarla. Esto evita un error muy común: olvidar normalizar en producción.

Aquí solo comprobamos que funciona:

In [ ]:
normalizar = layers.Rescaling(1.0 / 255)

muestra = normalizar(x_train[:5].astype("float32"))
print("Antes:   min =", x_train[:5].min(), "| max =", x_train[:5].max())
print("Después: min =", float(tf.reduce_min(muestra)), "| max =", float(tf.reduce_max(muestra)))

## 7. Data augmentation

El **data augmentation** genera variaciones aleatorias de las imágenes de entrenamiento (las voltea, las desplaza un poco, les aplica zoom...). Así el modelo ve imágenes «nuevas» en cada época y **generaliza mejor**, es decir, reduce el **overfitting**.

Elegimos transformaciones **realistas**: un barco volteado horizontalmente sigue siendo un barco, pero volteado verticalmente (boca abajo) no tiene sentido.

⚠️ El augmentation se aplica **solo en el entrenamiento**, nunca en validation ni en test. Las capas de Keras ya lo hacen automáticamente: solo se activan cuando `training=True`.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),         # volteo izquierda-derecha
    layers.RandomTranslation(0.1, 0.1),      # desplazamiento de hasta el 10 %
    layers.RandomZoom(0.1),                  # zoom de hasta el 10 %
], name="data_augmentation")

In [ ]:
# Visualizamos 8 versiones aumentadas de la misma imagen
imagen = x_train[7]

plt.figure(figsize=(14, 2.4))
plt.subplot(1, 9, 1)
plt.imshow(imagen)
plt.title("Original", fontsize=9)
plt.axis("off")
for i in range(8):
    aumentada = data_augmentation(imagen[np.newaxis].astype("float32"), training=True)
    plt.subplot(1, 9, i + 2)
    plt.imshow(aumentada[0].numpy().astype("uint8"))
    plt.title(f"Aumentada {i+1}", fontsize=9)
    plt.axis("off")
plt.suptitle(f"Data augmentation sobre un '{CLASS_NAMES[y_train[7]]}'")
plt.tight_layout()
plt.show()

## 8. Pipeline eficiente con `tf.data`

`tf.data` carga los datos por **lotes (batches)** y prepara el siguiente lote mientras la GPU procesa el actual (`prefetch`), lo que acelera el entrenamiento.

- `shuffle` → mezcla los datos de train en cada época (para que el modelo no aprenda el orden).
- `batch` → agrupa las imágenes de 64 en 64.
- `prefetch` → prepara los lotes por adelantado.

In [ ]:
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def crear_dataset(x, y, entrenamiento=False):
    """Convierte arrays de NumPy en un tf.data.Dataset listo para entrenar.

    Args:
        x: imágenes (N, 32, 32, 3) en uint8.
        y: etiquetas enteras (N,).
        entrenamiento: si es True, mezcla los datos (solo para train).
    """
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if entrenamiento:
        ds = ds.shuffle(buffer_size=len(x), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = crear_dataset(x_train, y_train, entrenamiento=True)
val_ds = crear_dataset(x_val, y_val)
test_ds = crear_dataset(x_test, y_test)

# Comprobamos un lote
imgs, labels = next(iter(train_ds))
print("Forma de un lote de imágenes:", imgs.shape)
print("Forma de un lote de etiquetas:", labels.shape)
print("Primeras etiquetas:", [CLASS_NAMES[i] for i in labels[:5].numpy()])

## ✅ Resumen de la fase 1

| Paso | Decisión | Por qué |
|---|---|---|
| División | 45k train / 5k val / 10k test, estratificada | Validation para early stopping; test intacto para la evaluación final |
| Normalización | `Rescaling(1/255)` **dentro del modelo** | Valores en [0,1] → entrenamiento estable; además el despliegue es más sencillo |
| Augmentation | Volteo horizontal, desplazamiento y zoom del 10 % | Reduce el overfitting; solo transformaciones realistas |
| Pipeline | `tf.data` con shuffle, batch 64 y prefetch | Entrenamiento más rápido |

👉 **Para el informe:** guarda las gráficas de este notebook (clic derecho → Guardar imagen); te servirán para la sección de preprocesamiento.

**Siguiente paso → Fase 2:** diseñar nuestra propia CNN.